# Data formats

In this notebook, you will have to read and write some data in particular formats : you will have to provide some python code or some answers to questions in a raw text format.

Make sure you are running this notebook on IGE jupyterhub in the pangeo-notebook environment otherwise the necessary python libraries will not be found.

Tip : duplicate this notebook in a draft version in which you can try and check the outputs with some prints and plots before submitting the final version to Github/Gitlab

In the submitted version of this notebook, make sure you do not add extra cell and make unnecessary prints as the automatic process will look for the content and output of the empty cells and check it against the expected answer

Note : 
Some data files are provided in this repository : ```/workdir/ping/alberta/ORL26-27/lab02```, copy them here in a ```data``` subdirectory and make sure that these data files will not be uploaded to Github/Gitlab by adding this line in your .gitignore :
```labs/02_data-management/data```

You can also prevent the upload of other files, like png files for instance by adding the line :
```labs/02_data-management/*png```

## Part 1 : Text files



Let's have a look first at the precipitation.txt file provided in the repo ```/workdir/ping/alberta/ORL26-27/lab02```

### 1.1  Using python open function, print the 4th line of the file 

Put your answer in the next cell, it should look like this :

```python
some python code
at some point, fourth_line=...
print(fourth_line)
````


### 1.2 Try to access the first element of the 4th line

### 1.3 What do you think is happenning ? 

To answer this question, type it in the next cell and do not modify its type which is defined as raw

### 1.4 Using numpy now, open the file and store the content in an array and print the first element of the 4th line of the file

### 1.5 Create a new txt file with precipitation in g/m2/s instead of kg/m2/s

The new file has to be written in the data subdirectory 

### 1.6 Do you think this data format allows to make our data FAIR and why ?

##  Part 2 Tabular data

Now we are going to have a look at precipitation.csv file which is a CSV file

Using python built-in csv reader, print the 4th line of the file

In [ ]:
import csv

with open('data/precipitation.csv', mode='r') as f:
    a = [line for line in csv.reader(f)]
print(a)        

In [ ]:
print(a[3])

In [ ]:
print(a[3][0])

Now using pandas

In [ ]:
import pandas
df = pandas.read_csv('data/precipitation.csv')
pandas.set_option('display.float_format', '{:.4g}'.format)
print(df)

In [ ]:
df.columns

In [ ]:
df.index

In [ ]:
df.loc[3]["Precip"]

Add a new column to the DataFrame with precip in g/m2/s and save it to a new csv file

In [ ]:
df["Precip in g"]=1000*df["Precip"]
print(df)

In [ ]:
df.to_csv("data/new_precipitation.csv")

How can you make this csv file more FAIR ?

## Part 3 Spatial data

Now we are going to work with the netcdf file sresa1b_ncar_ccsm3-example.nc

In [ ]:
from netCDF4 import Dataset
nc_f = 'data/sresa1b_ncar_ccsm3-example.nc'  # Your filename
nc_fid = Dataset(nc_f, 'r')  # Dataset is the class behavior to open the file
                             # and create an instance of the ncCDF4 class
print(nc_fid)
nc_fid.close()

In [ ]:
import xarray as xr
ds = xr.open_dataset('data/sresa1b_ncar_ccsm3-example.nc')
ds

In [ ]:
ds.ua.sel(lat=-70.73806, lon=355.78125, plev=10000.).values

In [ ]:
ds.ua.sel(lat=slice(-80,-70), lon=slice(350,360), plev=10000.)

In [ ]:
ds.ua.interp(lat=-70, lon=355, plev=10000.).values

In [ ]:
ds.pr.min(dim=["lat","lon"]).values

In [ ]:
ds.pr.max(dim=["lat","lon"]).values

In [ ]:
ds.pr.max(dim="lat").plot()

In [ ]:
ds.pr.plot()

Let's create our own netcdf file with xarray, following the CF conventions

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import xarray as xr

generate a random numpy array

In [ ]:
data = 283 + 5 * np.random.randn(5, 3, 4)
data

In [ ]:
times = pd.date_range('2018-01-01', periods=5)
times

In [ ]:
lons = np.linspace(-120, -60, 4)
lats = np.linspace(25, 55, 3)

In [ ]:
temp = xr.DataArray(data, coords=[times, lats, lons], dims=['time', 'lat', 'lon'])
temp

In [ ]:
temp.attrs['units'] = 'kelvin'
temp.attrs['standard_name'] = 'air_temperature'

temp

In [ ]:
pressure_data = 1000.0 + 5 * np.random.randn(5, 3, 4)
pressure = xr.DataArray(
    pressure_data, coords=[times, lats, lons], dims=['time', 'lat', 'lon']
)
pressure.attrs['units'] = 'hPa'
pressure.attrs['standard_name'] = 'air_pressure'

pressure

In [ ]:
ds = xr.Dataset(data_vars={'Temperature': temp, 'Pressure': pressure})
ds

In [ ]:
ds.lat.attrs['units'] = 'degree_north'
ds.lat.attrs['standard_name'] = 'latitude'

ds.lat

In [ ]:
ds.lon.attrs['units'] = 'degree_east'
ds.lon.attrs['standard_name'] = 'longitude'

ds.lon

In [ ]:
ds.to_netcdf("data/random_temp_press.nc")

check with ncdump in a terminal the content of the newly created file (load the pangeo-notebook environment with ```micromamba activate pangeo-notebook```